# Shravana — Full Pipeline on Colab

Run the complete subtitle translation pipeline end-to-end on a **T4 GPU**.
The FastAPI backend + full React UI are exposed via a Cloudflare tunnel.

**Prerequisites:**
- GPU runtime enabled: `Runtime → Change runtime type → T4 GPU`
- ~20 GB free disk (Colab provides ~78 GB)

**Branch:** `claude/optimize-qwen2-vl-VL8VC`

**Estimated setup time:** ~20 minutes (dominated by model downloads)

See `colab/COLAB_SETUP.md` for detailed documentation.

In [ ]:
# ── Cell 1: System packages ───────────────────────────────────────────────────
!apt-get install -y -q ffmpeg libmagic1

# Verify FFmpeg
!ffmpeg -version 2>&1 | head -1

# Node.js ships with Colab; confirm version ≥ 18
!node --version && npm --version

print('\n✓ System packages ready')

In [ ]:
# ── Cell 2: Clone / update repo ───────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/bakamono12/Shravana.git'
REPO_DIR  = '/content/Shravana'
BRANCH    = 'claude/optimize-qwen2-vl-VL8VC'

# Optional: HuggingFace token for gated models (leave blank if not needed)
HF_TOKEN  = ''  # e.g. 'hf_xxxxxxxxxxxx'

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)

if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}

!git -C {REPO_DIR} log --oneline -3
print('\n✓ Repo ready at', REPO_DIR)

In [ ]:
# ── Cell 3: Install Python dependencies ──────────────────────────────────────
# Order matters: torch first, then qwen-asr (pins transformers==4.57.6), then rest.

# 1. CUDA-enabled torch (Colab may already have it; this pins the CUDA version)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 2. bitsandbytes — needed for 4-bit quantization of Qwen2.5-VL
!pip install -q 'bitsandbytes>=0.43.0'

# 3. Backend package — installs all core deps including qwen-asr (pins transformers)
!pip install -q -e /content/Shravana/backend/

# 4. qwen-vl-utils — VLM image preprocessing utilities
!pip install -q 'qwen-vl-utils>=0.0.14'

# 5. torchcodec — video frame decoding (install after torch to get the right CUDA wheel)
!pip install -q torchcodec --extra-index-url https://download.pytorch.org/whl/cu121 || \
    pip install -q torchcodec

# 6. accelerate — required for device_map='auto' in Transformers
!pip install -q 'accelerate>=0.27.0'

print('\n✓ Python dependencies installed')

In [ ]:
# ── Cell 3b: Build React frontend ────────────────────────────────────────────
# The backend serves the built frontend when SHRAVANA_SERVE_UI=true.
# Opening the tunnel URL in a browser loads the full Shravana UI.

import subprocess, sys

frontend_dir = '/content/Shravana/frontend'

print('Installing npm dependencies...')
r = subprocess.run(['npm', 'install'], cwd=frontend_dir, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError('npm install failed')
print('npm install ✓')

print('Building frontend...')
r = subprocess.run(['npm', 'run', 'build'], cwd=frontend_dir, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise RuntimeError('npm run build failed')
print('npm build ✓')

from pathlib import Path
dist_index = Path('/content/Shravana/frontend/dist/index.html')
assert dist_index.exists(), f'Build output missing: {dist_index}'
print(f'\n✓ Frontend built → {dist_index}')

In [ ]:
# ── Cell 4: Write .env configuration ─────────────────────────────────────────
from pathlib import Path

env_content = '''# Shravana — Colab environment
STORAGE_DIR=/content/Shravana/storage
UPLOAD_DIR=/content/Shravana/storage/uploads
JOBS_DIR=/content/Shravana/storage/jobs
MODELS_DIR=/content/Shravana/storage/models

# Serve the built React UI from FastAPI (tunnel URL opens full UI)
SHRAVANA_SERVE_UI=true
BACKEND_HOST=0.0.0.0
BACKEND_PORT=8000

# VLM — 3B model with 4-bit quantization (~2.5 GB VRAM)
QWEN_VL_MODEL_ID=Qwen/Qwen2.5-VL-3B-Instruct
VLM_USE_4BIT=true
VLM_MAX_PIXELS=200704

# Pipeline
PIPELINE_CONCURRENCY=1
CHUNK_DURATION_SECONDS=60
CHUNK_OVERLAP_SECONDS=5
'''

env_path = Path('/content/Shravana/backend/.env')
env_path.write_text(env_content)

# Create storage directories
for d in ['storage/uploads', 'storage/jobs', 'storage/models']:
    (Path('/content/Shravana') / d).mkdir(parents=True, exist_ok=True)

print('✓ .env written to', env_path)
print('\nConfiguration:')
print(env_content)

In [ ]:
# ── Cell 5: Initialise database ───────────────────────────────────────────────
# Jupyter/Colab runs its own event loop, so asyncio.run() raises RuntimeError.
# Top-level `await` works natively in IPython ≥ 7 / all Colab kernels.
import sys
sys.path.insert(0, '/content/Shravana/backend')

from app.db import init_db
await init_db()

from pathlib import Path
db = Path('/content/Shravana/shravana.db')
assert db.exists(), f'Database not created at {db}'
print(f'✓ Database initialised: {db} ({db.stat().st_size:,} bytes)')

In [ ]:
# ── Cell 6: Download ML models (~10 GB, ~8-12 min) ───────────────────────────
# snapshot_download resumes partial downloads — safe to re-run if interrupted.

import sys
sys.path.insert(0, '/content/Shravana/backend')

from pathlib import Path
from huggingface_hub import snapshot_download
from app.ml.registry import MODEL_CONFIGS  # pulls repo IDs from config

MODELS_DIR = Path('/content/Shravana/storage/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Only two models in the minimal stack
MODELS_TO_DOWNLOAD = ['qwen3_asr', 'qwen2_5_vl']

for name in MODELS_TO_DOWNLOAD:
    cfg = MODEL_CONFIGS.get(name)
    if not cfg:
        print(f'⚠  Unknown model key: {name}')
        continue
    dest = MODELS_DIR / name
    if dest.exists() and any(dest.rglob('*.safetensors')):
        print(f'✓  {name}: already downloaded ({cfg["repo_id"]})')
        continue
    print(f'⬇  Downloading {name} — {cfg["repo_id"]} ...')
    snapshot_download(
        repo_id=cfg['repo_id'],
        local_dir=str(dest),
    )
    print(f'✓  {name}: done')

print('\n✓ All models ready')

In [ ]:
# ── Cell 7: Readiness check ───────────────────────────────────────────────────
# All checks must be green before starting the server.
import sys
sys.path.insert(0, '/content/Shravana')
sys.path.insert(0, '/content/Shravana/backend')

from colab.check_ready import run_checks
run_checks(raise_on_failure=True)  # raises SystemExit if any critical check fails

In [ ]:
# ── Cell 8: Start FastAPI server + Cloudflare tunnel ─────────────────────────
import os, re, subprocess, threading, time

os.chdir('/content/Shravana')
BACKEND_PORT = 8000
CF_BIN = '/content/cloudflared'

# ── Keep-alive: prevent Colab idle-timeout ────────────────────────────────────
def _keep_alive():
    while True:
        time.sleep(30)
        _ = sum(i * i for i in range(500))
threading.Thread(target=_keep_alive, daemon=True).start()
print('Keep-alive thread started')

# ── Download cloudflared if not present ──────────────────────────────────────
if not os.path.exists(CF_BIN):
    print('Downloading cloudflared ...')
    subprocess.run([
        'wget', '-q', '-O', CF_BIN,
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
    os.chmod(CF_BIN, 0o755)
    print('cloudflared downloaded ✓')

# ── Start uvicorn backend ────────────────────────────────────────────────────
env = os.environ.copy()
env['PYTHONPATH'] = '/content/Shravana/backend'

server_log = '/tmp/shravana_server.log'
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(BACKEND_PORT),
     '--log-level', 'info', '--no-access-log'],
    cwd='/content/Shravana/backend',
    env=env,
    stdout=open(server_log, 'w'),
    stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_proc.pid}  (logs → {server_log})')
time.sleep(5)  # wait for uvicorn to start

# Quick health check
import urllib.request
try:
    urllib.request.urlopen(f'http://localhost:{BACKEND_PORT}/api/system/health', timeout=5)
    print('Backend health check ✓')
except Exception as e:
    print(f'Health check warning: {e} — server may still be starting')

# ── Start Cloudflare tunnel ───────────────────────────────────────────────────
cf_log = '/tmp/cf_tunnel.log'
cf_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', f'http://localhost:{BACKEND_PORT}'],
    stdout=open(cf_log, 'w'), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log = open(cf_log).read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print('⚠  Tunnel URL not found — check /tmp/cf_tunnel.log')
else:
    print()
    print('=' * 60)
    print(f'  Open in browser → {public_url}')
    print(f'  API docs        → {public_url}/docs')
    print(f'  Health          → {public_url}/api/system/health')
    print('=' * 60)

In [ ]:
# ── Cell 9: End-to-end smoke test (optional) ──────────────────────────────────
# Uploads a generated 10-second sine-wave WAV and verifies the pipeline completes.
# No real audio needed — tests the full job lifecycle.

import io, struct, math, time, requests

BASE_URL = f'http://localhost:{BACKEND_PORT}'  # use local URL for the test

# Generate a minimal valid WAV file (10 s, 16kHz, mono, 440 Hz sine)
def _make_wav(duration_s: float = 10.0, sample_rate: int = 16000) -> bytes:
    n = int(duration_s * sample_rate)
    samples = [int(32767 * math.sin(2 * math.pi * 440 * i / sample_rate)) for i in range(n)]
    pcm = struct.pack(f'<{n}h', *samples)
    buf = io.BytesIO()
    # RIFF header
    data_size = len(pcm)
    buf.write(b'RIFF')
    buf.write(struct.pack('<I', 36 + data_size))
    buf.write(b'WAVE')
    buf.write(b'fmt ')
    buf.write(struct.pack('<IHHIIHH', 16, 1, 1, sample_rate, sample_rate * 2, 2, 16))
    buf.write(b'data')
    buf.write(struct.pack('<I', data_size))
    buf.write(pcm)
    return buf.getvalue()

wav_bytes = _make_wav()

print('Uploading test audio...')
resp = requests.post(
    f'{BASE_URL}/api/upload',
    files={'file': ('test_smoke.wav', wav_bytes, 'audio/wav')},
    data={'source_lang': 'en', 'target_lang': 'hi'},
)
resp.raise_for_status()
job = resp.json()
job_id = job.get('job_id') or job.get('id')
print(f'Job created: {job_id}')

# Poll until done (timeout 5 min)
deadline = time.time() + 300
while time.time() < deadline:
    status_resp = requests.get(f'{BASE_URL}/api/jobs/{job_id}')
    status = status_resp.json().get('status', 'unknown')
    print(f'  Status: {status}', end='\r')
    if status in ('done', 'completed', 'failed', 'error'):
        break
    time.sleep(5)

print(f'\nFinal status: {status}')
if status in ('done', 'completed'):
    subs = requests.get(f'{BASE_URL}/api/subtitles/{job_id}').json()
    print(f'\n✓ Smoke test passed — {len(subs)} subtitle entries returned')
else:
    print('⚠  Job did not complete — check server log at /tmp/shravana_server.log')

---
## Cell 10 — Reconnect after Colab inactivity

Run **only this cell** after Colab wakes from idle.
Models are still on disk — no re-download needed.

In [ ]:
# ── Cell 10: Reconnect after Colab inactivity ─────────────────────────────────
import os, re, subprocess, threading, time

os.chdir('/content/Shravana')
BACKEND_PORT = 8000
CF_BIN = '/content/cloudflared'

# Kill any stale processes
for name in ['uvicorn', 'cloudflared']:
    subprocess.run(['pkill', '-f', name], capture_output=True)
time.sleep(2)

# Keep-alive thread
def _keep_alive():
    while True:
        time.sleep(30)
        _ = sum(i * i for i in range(500))
threading.Thread(target=_keep_alive, daemon=True).start()
print('Keep-alive thread started')

# Restart uvicorn
env = os.environ.copy()
env['PYTHONPATH'] = '/content/Shravana/backend'

server_log = '/tmp/shravana_server.log'
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(BACKEND_PORT),
     '--log-level', 'info', '--no-access-log'],
    cwd='/content/Shravana/backend',
    env=env,
    stdout=open(server_log, 'w'),
    stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_proc.pid}')
time.sleep(5)

# Restart Cloudflare tunnel
cf_log = '/tmp/cf_tunnel.log'
cf_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', f'http://localhost:{BACKEND_PORT}'],
    stdout=open(cf_log, 'w'), stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    log = open(cf_log).read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print('⚠  Tunnel URL not found — check /tmp/cf_tunnel.log')
else:
    print()
    print('=' * 60)
    print(f'  Open in browser → {public_url}')
    print(f'  API docs        → {public_url}/docs')
    print('=' * 60)